In [21]:
import json

# 1. 기본 전체 보기 그래프 생성
bar_fig = viz.create_bar_chart()
scatter_fig = viz.create_scatter_plot()

# 2. 산점도 연동을 위한 자치구별 (생활인구, 정류소수) 데이터 좌표 추출
df_map = viz.prepare_map_data().sort_values("equity_score", ascending=True)
district_names = df_map["district_name"].tolist()
pop_list = df_map["living_population"].tolist()
stops_list = df_map["bus_stop_count"].tolist()

# 3. 막대 클릭 시 두 그래프가 동시 강조되는 통합 HTML 템플릿 생성
html_template = f"""
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>서울시 버스 인프라 형평성 시각화 대시보드</title>
    <style>
        body {{ font-family: 'Malgun Gothic', sans-serif; margin: 30px; background-color: #f8fafc; }}
        .card {{ background: white; padding: 25px; border-radius: 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.08); margin-bottom: 30px; }}
        h2 {{ color: #1e293b; border-bottom: 2px solid #e2e8f0; padding-bottom: 10px; margin-top: 0; }}
        p {{ color: #64748B; font-size: 14px; margin-bottom: 20px; }}
    </style>
</head>
<body>
    <div class="card">
        <h2>1. 자치구별 종합 형평성 점수 비교</h2>
        <p>💡 <b>막대를 마우스로 클릭해 보세요!</b> 해당 자치구가 파란색으로 강조되며, 아래 산점도에도 위치가 별표(★) 마커로 동시에 표시됩니다.</p>
        <div id="bar-chart-container">{bar_fig.to_html(full_html=False, include_plotlyjs='cdn')}</div>
    </div>

    <div class="card">
        <h2>2. 인구수 대비 버스 정류소 수 관계</h2>
        <div id="scatter-chart-container">{scatter_fig.to_html(full_html=False, include_plotlyjs=False)}</div>
    </div>

    <script>
        // 자치구 데이터 데이터셋 정의
        const districtNames = {json.dumps(district_names, ensure_ascii=False)};
        const popList = {json.dumps(pop_list)};
        const stopsList = {json.dumps(stops_list)};

        window.addEventListener('DOMContentLoaded', (event) => {{
            const barPlot = document.getElementById('bar-chart-container').getElementsByClassName('plotly-graph-div')[0];
            const scatterPlot = document.getElementById('scatter-chart-container').getElementsByClassName('plotly-graph-div')[0];

            barPlot.on('plotly_click', function(data){{
                if (!data || !data.points || data.points.length === 0) return;

                const pointIndex = data.points[0].pointIndex;
                const totalPoints = districtNames.length;
                const selectedName = districtNames[pointIndex];
                const selectedPop = popList[pointIndex];
                const selectedStops = stopsList[pointIndex];

                // 1) 막대그래프 강조: 선택 구는 파란색(#1D4ED8), 나머지는 연회색(#E2E8F0)
                const newBarColors = new Array(totalPoints).fill('#E2E8F0');
                newBarColors[pointIndex] = '#1D4ED8';
                Plotly.restyle(barPlot, {{'marker.color': [newBarColors]}});

                // 2) 산점도 강조: 기존 트레이스 유지 + 선택 구 별표(★) 트레이스 업데이트/추가
                const starTrace = {{
                    x: [selectedPop],
                    y: [selectedStops],
                    mode: 'markers',
                    marker: {{ size: 16, color: '#1D4ED8', symbol: 'star' }},
                    name: '선택 자치구',
                    hovertemplate: '<b>' + selectedName + ' (선택됨)</b><br>생활인구: %{{x:,.0f}}명<br>정류소: %{{y:,.0f}}개<extra></extra>'
                }};

                // 기존 별표 트레이스가 있으면 교체, 없으면 추가
                if (scatterPlot.data.length > 2) {{
                    Plotly.deleteTraces(scatterPlot, -1);
                }}
                Plotly.addTraces(scatterPlot, starTrace);
            }});
        }});
    </script>
</body>
</html>
"""

with open("interactive_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_template)

